# Bonus

🎯 84 feature’dan oluşan tam `ML_Houses_dataset.csv` dataset’iyle [buradan ulaşarak](https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv) serbestçe çalışabilirsiniz!

- Feature’ları inceleyin
- Uygun şekilde preprocess edin ve encode edin
- Feature engineering için beyin fırtınası yapın
- Bunları modelinize ekleyin
- Feature selection uygulayın

👇 Dosyayı yerel olarak `data` klasörüne kaydedin ve buradan içe aktarın.

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Veri setini data klasörüne indir
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv"

data_path = Path("data/ML_Houses_dataset.csv")
data_path.parent.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    pd.read_csv(url).to_csv(data_path, index=False)

# Veri setini yükle
df = pd.read_csv(data_path)

print("Veri setinin boyutu:", df.shape)
display(df.head())


Veri setinin boyutu: (1760, 85)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


ℹ️ Dataset’in açıklamasına mutlaka [buradan](https://drive.google.com/file/d/1qLxeQXufW_-KHOckpUweLPhitzjnP7H3/view?usp=sharing) referans verin.

In [2]:

# Veri setinin genel yapısı
print("VERİ SETİ BİLGİLERİ")
df.info()

# Eksik değer analizi
missing = pd.DataFrame({
    "Eksik Sayısı": df.isnull().sum(),
    "Eksik Oranı (%)": df.isnull().mean() * 100
})

missing = missing[missing["Eksik Sayısı"] > 0]
missing = missing.sort_values(
    "Eksik Oranı (%)", ascending=False
)

print("\nEKSİK DEĞER ANALİZİ")
display(missing)

# Veri tipleri
print("\nVERİ TİPLERİ")
print(df.dtypes.value_counts())

# Tekrarlanan satırlar
print("\nTekrarlanan satır:", df.duplicated().sum())


VERİ SETİ BİLGİLERİ
<class 'pandas.DataFrame'>
RangeIndex: 1760 entries, 0 to 1759
Data columns (total 85 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1760 non-null   int64  
 1   MSSubClass     1760 non-null   int64  
 2   MSZoning       1760 non-null   str    
 3   LotFrontage    1451 non-null   float64
 4   LotArea        1760 non-null   int64  
 5   Street         1760 non-null   str    
 6   Alley          112 non-null    str    
 7   LotShape       1760 non-null   str    
 8   LandContour    1760 non-null   str    
 9   Utilities      1760 non-null   str    
 10  LotConfig      1760 non-null   str    
 11  LandSlope      1760 non-null   str    
 12  Neighborhood   1760 non-null   str    
 13  Condition1     1760 non-null   str    
 14  Condition2     1760 non-null   str    
 15  BldgType       1760 non-null   str    
 16  HouseStyle     1760 non-null   str    
 17  OverallQual    1760 non-null   int64  
 18 

,Eksik Sayısı,Eksik Oranı (%)
WallMat,1755,99.715909
PoolQC,1751,99.488636
MiscFeature,1694,96.250000
Alley,1648,93.636364
Fence,1418,80.568182
MasVnrType,1062,60.340909
FireplaceQu,827,46.988636
LotFrontage,309,17.556818
GarageCond,93,5.284091
GarageYrBlt,93,5.284091



VERİ TİPLERİ
str        45
int64      35
float64     5
Name: count, dtype: int64

Tekrarlanan satır: 300


In [3]:

# Orijinal veriyi koruyalım
df_clean = df.copy()

# Id modelleme için kullanılmayacak
df_clean = df_clean.drop(columns=["Id"], errors="ignore")

# Özelliğin bulunmamasını ifade eden eksikleri doldur
none_cols = [
    "Alley", "PoolQC", "Fence", "MiscFeature",
    "MasVnrType", "FireplaceQu", "GarageType",
    "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2"
]

for col in none_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

# GarageYrBlt gibi diğer sayısal eksikleri
# medyan ile doldur
numeric_cols = df_clean.select_dtypes(
    include=["number"]
).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

# Kalan kategorik eksikleri en sık değerle doldur
categorical_cols = df_clean.select_dtypes(
    include=["object", "str", "category"]
).columns

for col in categorical_cols:
    mode = df_clean[col].mode(dropna=True)
    if not mode.empty:
        df_clean[col] = df_clean[col].fillna(mode.iloc[0])

print("Temizlenmiş veri boyutu:", df_clean.shape)
print("Kalan eksik değer:", df_clean.isnull().sum().sum())


Temizlenmiş veri boyutu: (1760, 84)
Kalan eksik değer: 0


In [4]:

# Temizlenmiş veri setini kopyala
df_fe = df_clean.copy()

# 1. Toplam yaşam ve bodrum alanı
df_fe["TotalArea"] = (
    df_fe["GrLivArea"] +
    df_fe["TotalBsmtSF"]
)

# 2. Evin yaşı (satış yılındaki)
df_fe["HouseAge"] = (
    df_fe["YrSold"] - df_fe["YearBuilt"]
).clip(lower=0)

# 3. Tadilattan sonra geçen yıl
df_fe["YearsSinceRemodel"] = (
    df_fe["YrSold"] - df_fe["YearRemodAdd"]
).clip(lower=0)

# 4. Toplam banyo sayısı
df_fe["TotalBathrooms"] = (
    df_fe["FullBath"] +
    0.5 * df_fe["HalfBath"] +
    df_fe["BsmtFullBath"] +
    0.5 * df_fe["BsmtHalfBath"]
)

# 5. Toplam veranda ve açık alan
df_fe["TotalPorchArea"] = (
    df_fe["OpenPorchSF"] +
    df_fe["EnclosedPorch"] +
    df_fe["3SsnPorch"] +
    df_fe["ScreenPorch"] +
    df_fe["WoodDeckSF"]
)

# 6. Garaj var mı?
df_fe["HasGarage"] = (
    df_fe["GarageArea"] > 0
).astype(int)

# 7. Bodrum var mı?
df_fe["HasBasement"] = (
    df_fe["TotalBsmtSF"] > 0
).astype(int)

# 8. Evin genel kalite ve yaşam alanı etkileşimi
df_fe["QualityArea"] = (
    df_fe["OverallQual"] *
    df_fe["GrLivArea"]
)

print("Önceki sütun sayısı:", df_clean.shape[1])
print("Yeni sütun sayısı:", df_fe.shape[1])

display(df_fe[[
    "TotalArea", "HouseAge",
    "TotalBathrooms", "HasGarage",
    "QualityArea"
]].head())


Önceki sütun sayısı: 84
Yeni sütun sayısı: 92


,TotalArea,HouseAge,TotalBathrooms,HasGarage,QualityArea
0,2566,5,3.5,1,11970
1,2524,31,2.5,1,7572
2,2706,7,3.5,1,12502
3,2473,91,2.0,1,12019
4,3343,8,3.5,1,17584


In [5]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Hedef değişken
y = df_fe["OverallQual"]

# Hedefi ve doğrudan hedef kullanılarak
# oluşturulan QualityArea sütununu çıkar
X = df_fe.drop(
    columns=["OverallQual", "QualityArea"]
)

# Sayısal ve kategorik özellikler
numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_cols),
    ("categorical", categorical_transformer, categorical_cols)
])

# Eğitim ve test bölünmesi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42
)

# Random Forest modeli
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Eğit ve değerlendir
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))



Accuracy: 0.7329545454545454
              precision    recall  f1-score   support

           2       1.00      0.50      0.67         2
           3       1.00      0.12      0.22         8
           4       0.65      0.38      0.48        29
           5       0.70      0.83      0.76       104
           6       0.71      0.76      0.73        85
           7       0.78      0.76      0.77        76
           8       0.74      0.72      0.73        32
           9       1.00      0.73      0.84        11
          10       1.00      1.00      1.00         5

    accuracy                           0.73       352
   macro avg       0.84      0.65      0.69       352
weighted avg       0.74      0.73      0.72       352



In [6]:

from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
import pandas as pd

# Ön işleme yalnızca eğitim seti üzerinde öğrenilir
X_train_encoded = model.named_steps["preprocessor"].transform(X_train)
X_test_encoded = model.named_steps["preprocessor"].transform(X_test)

# Modelin öğrendiği özellik önemleri
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importances = pd.Series(
    model.named_steps["classifier"].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("En önemli 20 özellik:")
display(importances.head(20).to_frame("Importance"))

# En önemli özelliklerden farklı gruplar oluştur
results = []

for n in [10, 20, 40, 60, 80]:
    n = min(n, len(feature_names))
    selected = importances.head(n).index

    selected_indices = [
        list(feature_names).index(col) for col in selected
    ]

    clf = RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        clf,
        X_train_encoded[:, selected_indices],
        y_train,
        cv=5,
        scoring="accuracy"
    )

    results.append({
        "Feature Count": n,
        "CV Accuracy": scores.mean()
    })

results_df = pd.DataFrame(results).drop_duplicates("Feature Count")

print("\nÖzellik sayısına göre model başarısı:")
display(results_df)


En önemli 20 özellik:


,Importance
numeric__SalePrice,0.051409
numeric__Pesos,0.048126
numeric__TotalArea,0.030224
numeric__GrLivArea,0.028092
numeric__YearBuilt,0.026595
numeric__HouseAge,0.025616
numeric__TotalBsmtSF,0.024922
numeric__1stFlrSF,0.023793
numeric__GarageYrBlt,0.023698
numeric__GarageArea,0.022700


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:81


Özellik sayısına göre model başarısı:


,Feature Count,CV Accuracy
0,10,0.639199
1,20,0.662641
2,40,0.671169
3,60,0.668325
4,80,0.668300


In [7]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# En iyi 40 özelliği seç
best_n = 40
selected_features = importances.head(best_n).index.tolist()

# Seçilen sütunların indekslerini belirle
selected_indices = [
    list(feature_names).index(col)
    for col in selected_features
]

# Eğitim ve test verilerini seç
X_train_selected = X_train_encoded[:, selected_indices]
X_test_selected = X_test_encoded[:, selected_indices]

# Son model
final_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train)

# Tahminler
final_predictions = final_model.predict(X_test_selected)

final_accuracy = accuracy_score(y_test, final_predictions)
final_f1 = f1_score(
    y_test, final_predictions, average="weighted"
)

print(f"Önceki model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Seçilmiş model Accuracy: {final_accuracy:.4f}")
print(f"Seçilmiş model Weighted F1: {final_f1:.4f}")
print(f"Kullanılan özellik sayısı: {best_n}")


Önceki model Accuracy: 0.7330
Seçilmiş model Accuracy: 0.7330
Seçilmiş model Weighted F1: 0.7266
Kullanılan özellik sayısı: 40
